# CEGMem — reproduce the paper on Colab

One command rebuilds `paper/main.pdf` from the shipped artifacts, running only what
is missing. This notebook is the Colab wrapper around `scripts/reproduce.sh`;
`RUNBOOK.md` in the repository is the authority on what each stage does.

**The rules (the same ones the script enforces):**

1. **Data present ⇒ nothing runs.** The decision is made per experiment *cell* from
   the merged `episodes.jsonl` (`scripts/grid_status.py`). With the artifacts
   installed, no model server is started and no oracle executes; the analysis
   chain (seconds) and the paper build are all that run.
2. **Data missing ⇒ the experiment runs**, for the missing cells only. That needs the
   ConDefects test data (`Test.zip`, several GB) and a proposer: Ollama on this
   runtime's GPU for `qwen2.5-coder:7b`, an API key for `gpt-4o-mini`.
   Model calls hit `cache/` first (`src.llm` memoises on prompt + nonce). Two
   regimes follow: with the artifacts (data + cache) the numbers come out
   bit-identical and nothing runs; without them the proposer is re-sampled at
   temperature 1.0, so the results are statistically similar, not identical.
   The shipped local run declares 47 of 216 `E9-freeguard` cells missing
   (`grid_coverage.json`); they are not run unless `FILL_DECLARED_GAPS=1`, and
   completing them moves the free-guarded table (then `--update-numbers`).
3. **Every paper number is regenerated and compared** with the committed
   `paper/common/*.json`; a difference fails the run unless `--update-numbers`.

Run the cells top to bottom. Every cell states what it reads, what it writes and
whether it can be skipped.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   configuration - nothing runs
# READS   nothing
# WRITES  python variables and a few environment variables
# SKIP?   NO
# ────────────────────────────────────────────────────────────────────────────
import os

# ── repository ──────────────────────────────────────────────────────────────
REPO_URL = "REPO_URL"                     # e.g. https://github.com/<you>/ceg-mem.git  (private: https://<TOKEN>@github.com/...)
BRANCH   = "promote-main"
WORKDIR  = "/content/ceg-mem"

# ── persistence: everything worth keeping lives on Drive ────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/ceg-mem"     # cache/, data/, logs/ are symlinked here

# ── the shipped artifacts (RUNBOOK.md "Artifacts") ──────────────────────────
# A path inside your mounted Drive (preferred - no quota, no confirm page), or a
# URL (https, or a Google Drive share link). Leave empty to run from scratch.
ARTIFACTS        = f"{DRIVE_ROOT}/artifacts.zip"
ARTIFACTS_SHA256 = ""                     # optional integrity check

# ── the ConDefects test data (several GB); needed only if a cell must RUN ───
TEST_ZIP_DRIVE_PATH = f"{DRIVE_ROOT}/Test.zip"    # or None

# ── proposers ───────────────────────────────────────────────────────────────
PROPOSERS = "local cloud"                 # which blocks to run; "local" alone is fine
INSTALL_OLLAMA = False                    # True only if a LOCAL cell must run (needs a GPU runtime)
LLM_API_KEY = ""                          # only if a CLOUD cell must run (gpt-4o-mini)

for k, v in {"PROPOSERS": PROPOSERS, "ARTIFACTS": ARTIFACTS, "ARTIFACTS_SHA256": ARTIFACTS_SHA256}.items():
    os.environ[k] = v
if LLM_API_KEY:
    os.environ["LLM_API_KEY"] = LLM_API_KEY
print("configured")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   mount Drive
# READS   your Drive
# WRITES  {DRIVE_ROOT}/cache, /data, /logs
# SKIP?   NO - without it nothing survives the session
# ────────────────────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
import os
for sub in ("cache", "data", "logs"):
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("persistent root:", DRIVE_ROOT)
!ls -la {DRIVE_ROOT}

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   clone the repository (branch promote-main) and point cache/ data/ logs/ at Drive
# READS   GitHub, {DRIVE_ROOT}
# WRITES  /content/ceg-mem, three symlinks
# SKIP?   no
# ────────────────────────────────────────────────────────────────────────────
import os
os.chdir('/content')
!test -d {WORKDIR} || git clone --branch {BRANCH} {REPO_URL} {WORKDIR}
os.chdir(WORKDIR)
!git fetch --quiet && git checkout {BRANCH} && git pull --ff-only
!git log --oneline -1

# data/mutants.py is tracked source and must exist inside the Drive-backed data/
!cp -n data/mutants.py {DRIVE_ROOT}/data/ 2>/dev/null; true
!rm -rf cache data logs
!ln -s {DRIVE_ROOT}/cache cache && ln -s {DRIVE_ROOT}/data data && ln -s {DRIVE_ROOT}/logs logs
!ls -la | grep -E ' (cache|data|logs) '

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   python dependencies
# READS   requirements.txt
# WRITES  site-packages
# SKIP?   yes on a warm runtime (seconds to confirm)
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!pip install -q -r requirements.txt
!python3 -c "import openai, dotenv, numpy, scipy, matplotlib; print('deps ok')"
# scripts/reproduce.sh creates .venv only when it is absent; on Colab the system
# python already has everything, so give it one that resolves to it.
!test -d .venv || python3 -m venv --system-site-packages .venv

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   the benchmark: ConDefects code (125 MB clone) + Test.zip from Drive
# READS   GitHub, {TEST_ZIP_DRIVE_PATH}
# WRITES  external/ConDefects/{Code,Test}
# SKIP?   the clone: no (fit_theory/anchoring/patch_quality read source files).
#         Test.zip: yes when every cell is a hit - it is needed only to RUN cells
#         or to recompute a Tier-B artifact.
# ────────────────────────────────────────────────────────────────────────────
import os, shutil, zipfile; os.chdir(WORKDIR)
!python3 scripts/fetch_condefects.py || true          # exits 1 while Test/ is absent: that is the report

dst = "external/ConDefects/Test.zip"
if os.path.isdir("external/ConDefects/Test"):
    print("Test/ already unpacked")
elif TEST_ZIP_DRIVE_PATH and os.path.exists(TEST_ZIP_DRIVE_PATH):
    print("copying Test.zip from Drive ...")
    shutil.copyfile(TEST_ZIP_DRIVE_PATH, dst)
    if not zipfile.is_zipfile(dst):
        raise SystemExit(f"{dst} is not a zip archive - an HTML error page saved under the name?")
    !python3 scripts/fetch_condefects.py                # unpacks and verifies
else:
    print("no Test.zip - fine as long as every cell is a hit (see the dry run below)")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   proposer, if a cell must run
# READS   ollama.com (optional), LLM_API_KEY (optional)
# WRITES  /usr/local/bin/ollama + the model weights
# SKIP?   yes when the grid is complete (the dry run below says so)
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
if INSTALL_OLLAMA:
    !nvidia-smi -L || echo "NO GPU - Runtime > Change runtime type > GPU, then rerun"
    !bash scripts/install_ollama_colab.sh              # installs + pulls qwen2.5-coder:7b; serves nothing
else:
    print("INSTALL_OLLAMA=False - no local proposer on this runtime")
print("cloud key:", "set" if os.environ.get("LLM_API_KEY") else "not set (needed only if a gpt-4o-mini cell must run)")

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   install the artifacts, then print the plan
# READS   ARTIFACTS
# WRITES  cache/, data/<run>/, logs/<run>/ (never overwrites an existing file)
# SKIP?   the install is idempotent; the dry run costs seconds
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
!bash scripts/reproduce.sh --stage artifacts
!bash scripts/reproduce.sh --dry-run
# Every preset should read "hit". A preset marked RUN needs Test.zip and a
# proposer (cells above) before the next cell can finish.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   the one command
# READS   data/<run>/, cache/
# WRITES  data/<run>/ (Tier C), paper/common/*.json (only if identical), paper/main.pdf
# TIME    minutes with the artifacts; hours to days from scratch
# ────────────────────────────────────────────────────────────────────────────
import os; os.chdir(WORKDIR)
# --no-paper if the runtime has no TeX; add --update-numbers ONLY to accept a
# deliberate change of the reference numbers.
!apt-get -qq install -y texlive-latex-extra texlive-publishers texlive-fonts-extra texlive-plain-generic latexmk > /dev/null 2>&1 || echo "TeX install failed - the paper stage will be skipped with a warning"
!bash scripts/reproduce.sh

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
# STAGE   keep the result
# READS   paper/main.pdf
# WRITES  {DRIVE_ROOT}/main.pdf
# ────────────────────────────────────────────────────────────────────────────
import os, shutil; os.chdir(WORKDIR)
if os.path.exists("paper/main.pdf"):
    shutil.copyfile("paper/main.pdf", f"{DRIVE_ROOT}/main.pdf")
    print("copied paper/main.pdf ->", f"{DRIVE_ROOT}/main.pdf")
else:
    print("paper/main.pdf not built - see the summary table above")
!git status --short paper/common/*.json   # empty when the regenerated numbers were identical